In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
pip install youtube-transcript-api


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

Step 1.1 Document Ingestion

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "7ARBJQn6QkM"


ytt_api.fetch(video_id)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="At some point, you have to believe something.\xa0\nWe've reinvented computing as we know it. What", start=0.08, duration=3.68), FetchedTranscriptSnippet(text='is the vision for what you see coming next? We\xa0\nasked ourselves, if it can do this, how far can', start=3.76, duration=4.72), FetchedTranscriptSnippet(text='it go? How do we get from the robots that\xa0\nwe have now to the future world that you', start=8.48, duration=4.64), FetchedTranscriptSnippet(text='see? Cleo, everything that moves will be\xa0\nrobotic someday and it will be soon. We', start=13.12, duration=4.08), FetchedTranscriptSnippet(text="invested tens of billions of dollars before\xa0\nit really happened. No that's very good, you", start=17.2, duration=5.04), FetchedTranscriptSnippet(text='did some research! But the big breakthrough\xa0\nI would say is when we...', start=22.24, duration=5.994), FetchedTranscriptSnippet(text="That's Jensen Huang, and whethe

In [ ]:

video_id = "7ARBJQn6QkM"
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, )

    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

At some point, you have to believe something. 
We've reinvented computing as we know it. What is the vision for what you see coming next? We 
asked ourselves, if it can do this, how far can it go? How do we get from the robots that 
we have now to the future world that you see? Cleo, everything that moves will be 
robotic someday and it will be soon. We invested tens of billions of dollars before 
it really happened. No that's very good, you did some research! But the big breakthrough 
I would say is when we... That's Jensen Huang, and whether you know it or not
his decisions are shaping your future. He's the CEO of NVIDIA, the company that skyrocketed over the past few
years to become one of the most valuable companies in the world because they led a fundamental shift 
in how computers work unleashing this current explosion of what's possible with technology. 
"NVIDIA's done it again!" We found ourselves being one of the most important technology companies in 
the world and potentiall

Step 1.2 - Indexing (Text Splitting)

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])
len(chunks)
chunks[10]

Document(metadata={}, page_content="oh my god it's revolutionizing all of these other industries as well, it's beginning to change\xa0\nhow we see what's possible with computers and my understanding is that in the early 2000s you\xa0\nsee this and you realize that actually doing that is a little bit difficult because what that\xa0\nresearcher had to do is he had to sort of trick the GPUs into thinking that his problem was a\xa0\ngraphics problem. That's exactly right, no that's very good, you did some research. So you create\xa0\na way to make that a lot easier. That's right Specifically it's a platform called CUDA which\xa0\nlets programmers tell the GPU what to do using programming languages that they already know like\xa0\nC and that's a big deal because it gives way more people easier access to all of this computing\xa0\npower. Could you explain what the vision was that led you to create CUDA? Partly researchers\xa0\ndiscovering it, partly internal inspiration and and partly solvin

Step 1.3 and 1.4 - Indexing ( Embedding generation and storing in Vector store )

In [30]:

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

c:\Users\Shankar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5778.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
vector_store.index_to_docstore_id

{0: 'a8b3b118-efe6-4a02-85cd-5c3044415032',
 1: 'c8d8a7a3-0103-45cf-bceb-008bd28a391a',
 2: '331eb4ab-1c9a-48ee-a4dc-0f9fc998a207',
 3: 'e16ebc75-1100-4713-b61b-7e2a4649b10f',
 4: 'd49dc00f-134c-4c84-8b92-5566abbb2592',
 5: '690bdd35-832a-4a1a-a413-ca2f2117656e',
 6: '82ba1a02-3433-4f88-b75e-f2a9ece8d2f0',
 7: '283d52d1-1adc-47e1-831b-8f9bf3c46eda',
 8: '652684ad-5088-41c8-bd03-57aa35235cda',
 9: '4e4ee3ad-21fb-4ebc-8b6f-7cea34c94b6c',
 10: '859e721a-c79e-4519-9b8b-3df63eb6b9b5',
 11: 'b5dc37c1-7136-4f12-a1e4-1f73a0e0c63a',
 12: '57d0c5f1-d633-44ba-bbd3-8d2a8a53760b',
 13: '5c033d52-3ed3-4540-a6a6-5b7ea4ed3836',
 14: '6105739b-5481-490b-80ae-729c78428a1a',
 15: 'a01d2e48-8ee0-4d98-9545-236c7c46ee65',
 16: '038bb016-37e2-442b-8e0c-27b344c4bf82',
 17: '5fc273e8-98cd-4016-84cd-f93626cdc702',
 18: '6f029a6a-1819-49e6-a609-81edf62e122d',
 19: '70653ef6-13a6-41aa-a188-965cda92a3ce',
 20: 'bb74df9a-eeef-481f-901c-25b1401c50be',
 21: 'ba78766a-4aeb-4a1a-bc94-fdf5542c3aed',
 22: 'a58c7adc-5296-

In [33]:
vector_store.get_by_ids(['99cbb221-8959-4535-a13b-6e3187fcc8d4'])

[Document(id='99cbb221-8959-4535-a13b-6e3187fcc8d4', metadata={}, page_content="place. That if we wanted to drive we can drive but otherwise you know take a nap or enjoy\xa0\nyour car like it's a home theater of yours, you know read from work to home and at that\xa0\npoint you're hoping that you live far away and so you could be in a car for longer.\xa0\nAnd you look back and you realize that there's this company almost at\xa0\nthe epicenter of all of that and happens to be the company that\nyou grew up playing games\xa0with. I hope for that to be\nwhat the next generation learn. Thank you so much\xa0for your time.\nI enjoyed it, thank you! I'm glad!")]

Step 2 : Retrieval

In [35]:
retriever = vector_store.as_retriever (search_type = "similarity", search_kwargs={"k":4})

In [36]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000029F629E92B0>, search_kwargs={'k': 4})

In [37]:
retriever.invoke('What is the impact of AI')

[Document(id='15aa11c4-1403-462d-9234-232356258cd4', metadata={}, page_content="somebody. You know for example self-driving car that wants to drive nicely and drive properly\xa0\nand just somehow the sensor broke down or it didn't detect something. Or you know made it\xa0\ntoo aggressive turn or whatever it is. It did it poorly. It did it wrongly. And so that's\na whole bunch of engineering that has to be done to to make sure that AI safety is upheld\xa0\nby making sure that the product functions properly. And then and then lastly you know whatever what\xa0\nhappens if the system, the AI wants to do a good job but the system failed. Meaning the AI wanted\xa0\nto stop something from happening and it turned out just when it wanted to do\xa0\nit, the machine broke down. And so this is no different than a flight computer inside\xa0\na plane having three versions of them and then so there's triple redundancy inside the\xa0\nsystem inside autopilots and then you have two pilots and then you 

Step 3 - Augmentation

In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "Private_Key"

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="",
    temperature=0.5,
    max_new_tokens=512
)

In [82]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    task="conversational",
    temperature=0.5,
    max_new_tokens=512
)

model = ChatHuggingFace(llm = llm)

In [83]:
prompt = PromptTemplate(
    template = """
      You are a helpful AI assistant. Answer ONLY from the following provided transcript context.
      If the context is insufficient, just say you don't know. 
            
      {context}
       Question : {question} 
    input_variables = ['context'. ['question']]
"""
)

In [94]:
question = "What are we going to have in the next 10 years?"
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='9272c5a7-3b61-42ee-85fb-94948610dca1', metadata={}, page_content="it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technol

In [95]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technology? How can\n\ntalked about CUDA, we've talked about bets you've made in AI - 

In [96]:
final_prompt = prompt.invoke({"context":context_text, "question":question})

In [97]:
final_prompt

StringPromptValue(text="\n      You are a helpful AI assistant. Answer ONLY from the following provided transcript context.\n      If the context is insufficient, just say you don't know. \n\n      it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the

Step 4 = Generation

In [98]:
answer = model.invoke(final_prompt)
print(answer.content)

You're referring to the context of the conversation, which involves a discussion about the potential future of human-robot interaction, specifically with the integration of Omniverse and Cosmos technologies.

In the next 10 years, it's likely that:

1. Humans will interact with robots in increasingly robotic environments, such as:
   - Autonomous vehicles (cars, trucks, drones)
   - Lawn mowers and other household robots
   - Industrial robots for manufacturing and assembly
   - Personal robots for daily tasks and companionship
   - Robotic exoskeletons and prosthetics for rehabilitation and assistance
2. Robots will become more advanced and sophisticated, with capabilities such as:
   - Advanced sensing and perception
   - Enhanced intelligence and decision-making
   - Ability to learn and adapt to new situations
3. The integration of Omniverse and Cosmos technologies will lead to the creation of:
   - New types of generative worlds and multiverse systems
   - More realistic and immer

Building a Chain

In [156]:
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [157]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [186]:
parallel_chain =RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [181]:
parallel_chain.invoke("What are we going to have in the next 10 years?")

{'context': "it opened up this universe of opportunities and universe of problems that we can go solve. And\xa0\nthat gets us quite excited. It feels like we are on the cusp of this truly enormous change.\xa0\nWhen I think about the next 10 years, unlike the last 10 years, I know we've gone through a lot of\xa0\nchange already but I don't think I can predict anymore how I will be using the technology that is\xa0\ncurrently being developed. That's exactly right. I think the last 10, the reason why you feel that way\xa0\nis, the last 10 years was really about the science of AI. The next 10 years we're going to have plenty\xa0\nof science of AI but the next 10 years is going to be the application science of AI. The fundamental\xa0\nscience versus the application science. And so the the applied research, the application side of AI\xa0\nnow becomes: How can I apply AI to digital biology? How can I apply AI to climate technology? How can\n\ntalked about CUDA, we've talked about bets you've m

In [187]:
parser = StrOutputParser()

In [188]:
final_chain = parallel_chain | prompt |model | parser



In [189]:
result = final_chain.invoke('What are we going to have in the next 10 years?')

In [190]:
print(result)

You're looking for information on what people might expect to see in the next 10 years, given the context of the conversation. 

The speaker seems to be excited about the potential future of robotics and artificial intelligence, particularly in the areas of digital biology, climate technology, and human-robot interaction. They mention that the next 10 years will be the application science of AI, which suggests that they anticipate significant advancements in these areas.

The speaker also mentions the potential for robots to become increasingly autonomous and integrated into daily life, with the possibility of robots taking over mundane tasks such as driving or performing household chores. They also mention the potential for robots to learn and adapt to their environment, which could lead to a future where robots are indistinguishable from humans.

In terms of specific predictions, the speaker mentions that they expect the next 10 years to be the "cusp of this truly enormous change" in